# Locality-Sensitive Hashing — Collision Probability and the ρ Exponent

A narrative companion to `locality_sensitive_hashing.py`, the canonical reference that *owns the
numbers*. IVF cut the space into Voronoi cells; the graph topics walked a graph. LSH is the third ANN
family and the only one with a sharp, distribution-free theory of its own: hash so that **near points
collide more often than far ones**, and build the index on that collision probability. The centerpiece
is random-hyperplane **SimHash** — which is signed random projection, so this topic *imports* the
Johnson–Lindenstrauss projection machinery rather than reinventing it.

Three movements, each an assertion in the `.py`:
1. the family and its collision probability `P[h(x)=h(y)] = 1 − θ/π`;
2. amplification — the AND/OR construction and the S-curve `g(p) = 1 − (1−pᵏ)ᴸ`;
3. the exponent `ρ = ln(1/p₁)/ln(1/p₂) < 1`, sublinear query time, and the cross-index head-to-head.

This notebook imports the module and runs its tests section by section. It stores no outputs; run it
top to bottom (`jupyter execute` or *Run All*).

In [ ]:
import sys, pathlib

HERE = pathlib.Path.cwd()
for cand in (HERE, HERE / "notebooks" / "locality-sensitive-hashing"):
    if (cand / "locality_sensitive_hashing.py").exists():
        sys.path.insert(0, str(cand))
        break

import locality_sensitive_hashing as lsh

## Movement 1 — the family and its collision probability

A hash family is *(r₁, r₂, p₁, p₂)-sensitive* if it collides with probability ≥ p₁ inside radius r₁ and
≤ p₂ beyond r₂. For SimHash — one bit per random hyperplane, `b(x) = 1[⟨x, h⟩ ≥ 0]` with `h ~ N(0, I)` —
the collision probability of two vectors at angle θ is **exactly** `1 − θ/π`: a hyperplane separates
them iff its normal falls in the wedge between them, of angular measure θ/π (Goemans–Williamson /
Charikar).

In [ ]:
lsh.test_collision_probability_matches_theory()

for r in lsh.collision_curve(lsh.THETA_FRACS, d=128, n_planes=8000, seed=0):
    print(f"  theta/pi={r['theta_frac']:.1f}:  empirical={r['empirical']:.4f}   1 - theta/pi={r['theory']:.2f}")

**The byte-for-byte collapse anchor.** Before amplifying, we pin that the amplified family at
`k = L = 1` *is* the bare SimHash: from one shared array of per-plane agreements,
`amplified_collision_from_matches(·, 1, 1)` equals the base collision rate exactly, and the S-curve at
`k = L = 1` is the identity `g(p) = p`.

In [ ]:
lsh.test_single_hash_collapse()

## Movement 2 — amplification: AND/OR and the S-curve

One bit is a weak filter. Concatenate **k** bits (an AND: a table-collision needs all k, probability
`pᵏ`) and union **L** independent tables (an OR: collide if any table matches, `1 − (1 − pᵏ)ᴸ`). The
composite `g(p) = 1 − (1 − pᵏ)ᴸ` is the canonical LSH **S-curve**; (k, L) place and sharpen its
threshold, trading recall against candidate-set size. We verify the empirical amplified collision
tracks `g(p)`, and that raising k sharpens while raising L lifts.

In [ ]:
lsh.test_s_curve_sharpens()

sc = lsh.s_curve_table(lsh.S_CURVE_K, lsh.S_CURVE_L, (0.5, 0.7, 0.85))
for cur in sc["curves"]:
    vals = ", ".join(f"g({p})={g}" for p, g in zip(sc["p"], cur["g"]))
    print(f"  k={cur['k']:>2} L={cur['L']:>2}:  {vals}")

## Movement 3 — the ρ exponent, sublinear query time, and the head-to-head

Tuning (k, L) to the gap (p₁, p₂) yields a (c, r)-ANN data structure with query time `O(nᵖ)` and space
`O(n^{1+ρ})`, where **ρ = ln(1/p₁)/ln(1/p₂) < 1** whenever p₁ > p₂ — sublinear, by a *data-independent*
family. ρ shrinks as the approximation factor c widens the near/far gap.

In [ ]:
lsh.test_rho_below_one()

for r in lsh.rho_curve(lsh.RHO_THETA1_FRAC, lsh.C_GRID):
    print(f"  c={r['c']:>4}:  p1={r['p1']:.3f}  p2={r['p2']:.3f}  ->  rho={r['rho']:.4f}")

The LSH index itself: L tables of k-bit SimHash, query by the union of colliding buckets, exact-rank
the candidates. Its speed/recall frontier sweeps the number of tables L — more tables widen the
candidate union (more recall, more cost).

In [ ]:
lsh.test_lsh_recall_cost_tradeoff()

**The cross-index head-to-head (measured on this cloud — no universal winner).** LSH vs IVF vs
flat-NSW vs HNSW on **one** normalized cloud and **one** shared ground truth, by distance computations
per query. We L2-normalize so SimHash's angular ranking equals the Euclidean ground truth the graph/IVF
indexes are scored against. Cost is honest: LSH pays for its `L·k` hashing projections (the analogue of
IVF's `nlist` coarse comparisons) plus one exact distance per candidate. On this low-rank cloud the
data-aware indexes dominate the data-oblivious hash — the price of a distribution-free guarantee.

In [ ]:
lsh.test_head_to_head()

## The numbers the laboratory mirrors

`viz_constants()` prints exactly what `LocalitySensitiveHashingLaboratory.tsx` bakes to the decimal:
the collision curve (Panel A), the S-curves (Panel B), and the ρ curve + head-to-head frontiers
(Panel C).

In [ ]:
lsh.viz_constants()
print("All checks passed.")